# boston crime analysis
Adapted from supplied coursework credited to Vedant Devidas Patil. Maintained for ds2204. See ATTRIBUTION.md. Historical reports are preserved separately and may contain superseded conclusions.

In [ ]:
#importing the required libraries and packages
import glob
#We can use glob to search for a specific file pattern, 
#or perhaps more usefully, search for files where the filename matches a certain pattern by using wildcard characters

import os
#The OS module in Python provides functions for 
#creating and removing folder, fetching its contents, changing and identifying the current directory

import pandas as pd

In [ ]:
from pathlib import Path
cwd = Path.cwd()
assert (cwd / "df_2015.csv").exists(), "Run from the repository root."

In [ ]:
file_list = sorted(cwd.glob("df_20[12][0-9].csv"))
[p.name for p in file_list]

In [ ]:
df_append = pd.concat([pd.read_csv(path, low_memory=False) for path in file_list], ignore_index=True)
df_append.head()

In [ ]:
main_df = df_append

#Displays the first 50 rows
main_df.head(50)

In [ ]:
#Displays the last 50 rows
main_df.tail(50)

In [ ]:
#selecting all the duplicate rows in the dataframe including the first occurance.
duplicate = main_df[main_df.duplicated(keep=False)]
duplicate

In [ ]:
#Dropping the duplicates from the dataset keeping the first occurance.
main_df = main_df.drop_duplicates()

In [ ]:
#The info() contains the number of columns, column labels, column data types, memory usage, 
#range index, and the number of cells in each column

main_df.info()

In [ ]:
#By observing the variables, it can be easily deduced that the Location variable 
#is just storing the concatenation of the Lat and Long, 
#so let's get rid of this.
df = main_df.drop("Location", axis = 1)
df.head()

In [ ]:
main_df.describe()

#The describe() method returns description of the data in the DataFrame.
#If the DataFrame contains numerical data, the description contains these information for each column:
# count - The number of not-empty values.
# mean - The average (mean) value.
# std - The standard deviation.
# min - the minimum value.
# 25% - The 25% percentile*.
# 50% - The 50% percentile*.
# 75% - The 75% percentile*.
# max - the maximum value.

In [ ]:
#next we have variables like INCIDENT_NUMBER and OFFENSE_CODE, 
#which serve no purpose in answering our questions. So, let's get rid of them too.

df = df.drop(["INCIDENT_NUMBER", "OFFENSE_CODE"], axis = 1)
df.head(5)

In [ ]:
#The column SHOOTING appears to have nothing but NaN values. Let's confirm that.
df['SHOOTING'].describe()

In [ ]:
#Counting the number of null values in the dataframe
df["SHOOTING"].isna().sum()

In [ ]:
# Missing shooting flags remain unknown; do not impute a confirmed negative.
df["SHOOTING"].value_counts(dropna=False)

In [ ]:
#Again counting the number of null values after replacing NAN with N
df.OFFENSE_CODE_GROUP.isna().sum()

In [ ]:
#Checking which months are included for which particular year.

years = [2015, 2016, 2017, 2018]
for year in years:
    print(df[df["YEAR"] == year]["MONTH"].unique())
    
#Here, we can see that 2015 has data from June to Dec whereas rest all year have all 12 months data.

In [ ]:
#Checking missing/null values in all the below columns

time_list = ["HOUR", "DAY_OF_WEEK", "MONTH", "OCCURRED_ON_DATE", "DISTRICT", "REPORTING_AREA", "STREET", "Lat", "Long"]
for item in time_list:
    print("No. of NaNs in {} = {}".format(item, df[item].isna().sum()))
    
#DISTRICT and STREET have missing values. REPORTING_AREA looks like a good variable to consider, 
#since it has no missing values.

In [ ]:
#let's also have a look at Lat and Long
df["Long"].describe()

In [ ]:
#Lat is assigned to -1 to indicate an absence. Similarly, for Long, -1 indicates absence. Let's replace -1 with None.

In [ ]:
df["Lat"] = df["Lat"].replace(-1, float("nan"))
df["Long"] = df["Long"].replace(-1, float("nan"))
df["Long"].isna().sum()

In [ ]:
#Q1 Which are the different types of crimes that are most and least common?

In [ ]:
#First, we perform group-by on the offence code group column as the crimes description/names are mentioned in that column
#size() function return the number of rows if Series. 
#Otherwise it returns the number of rows times number of columns if DataFrame

#The sort_values() function sorts the dataframe column in ascending or descending order.
crime_count = pd.DataFrame(df.groupby('OFFENSE_CODE_GROUP').size().sort_values(ascending=False).rename('Count').reset_index())


crime_count
#Displaying the dataframe.

In [ ]:
# These are recorded offense rows, not unique incidents or population-adjusted risks.
crime_count.head(10)

In [ ]:
#importing the matplotlib package for displaying bar plot.
import matplotlib.pyplot as plt

In [ ]:
#Since, the dataframe contains 67 columns, I selected just top 10 crimes that happens the most.

crime_count[:10].plot(x='OFFENSE_CODE_GROUP',y='Count',kind='bar')

In [ ]:
# Which streets have the most and fewest recorded offense rows?

In [ ]:
#Grouping the data using the street column and arranging them in ascending order.
#Renaming the newly created column as Count, which displays the count of crimes happening on those particular streets.
crime_location = pd.DataFrame(df.groupby('STREET').size().sort_values(ascending=False).rename('Count').reset_index())

#Dsiplaying the top 5 streets that have the most cost.
crime_location.head()

In [ ]:
crime_location = pd.DataFrame(df.groupby('STREET').size().sort_values(ascending=True).rename('Count').reset_index())

#Displaying the top 5 streets with least crimes.
crime_location.head()

In [ ]:
# Examine explicitly flagged shooting records separately.

In [ ]:
shootings = main_df[main_df["SHOOTING"].eq("Y")]
shootings.head()

In [ ]:
#Counting the nnumber of shooting in each top 5 streets.
shooting_location = pd.DataFrame(shootings.groupby('STREET').size().sort_values(ascending=False).rename('Count').reset_index())
shooting_location.head()

In [ ]:
# Counts alone cannot establish how safe a street is to live on.
# Reporting practices, exposure, population and multiple offenses per incident matter.

In [ ]:
#Q3 Monthly Crime Activity from 2015 - 2018 / Does the frequency of crimes change over the Year?

In [ ]:
#A pivot table usually consists of row, column, and data fields and arranges them accordingly
crime_activity_plot = pd.DataFrame(df.groupby(['MONTH','YEAR']).size().sort_values(ascending=False).rename('Count').reset_index())


crime_activity_2015_2018 = crime_activity_plot.pivot_table(values='Count',index='MONTH',columns='YEAR')

In [ ]:
crime_activity_2015_2018

In [ ]:
import seaborn as sns

#Generating heatmap using seaborn.
sns.heatmap(crime_activity_2015_2018)

In [ ]:
# Compare coverage before interpreting annual totals: 2015 starts mid-year.
# These descriptive counts do not establish changes in underlying crime risk.
main_df.groupby("YEAR")["OCCURRED_ON_DATE"].agg(["min", "max", "count"])

In [ ]:
#Q4 Check the trend of crimes across all 4 seasons for all years.

In [ ]:
#accessing the data from 12,1,2 months and storing it in winter variable
winter = pd.DataFrame(df[(df.MONTH.isin([12,1,2]))])
winter_data = len(winter.index)

#accessing the data from 3,4,5 months and storing it in spring variable
spring = pd.DataFrame(df[(df.MONTH.isin([3,4,5]))])
spring_data = len(spring.index)

summer = pd.DataFrame(df[(df.MONTH.isin([6,7,8]))])
summer_data = len(summer.index)

fall = pd.DataFrame(df[(df.MONTH.isin([9,10,11]))])
fall_data = len(fall.index)

In [ ]:
season_df = pd.DataFrame({ 'Winter Crime Count': [winter_data], 'Spring Crime Count': [spring_data], 'Summer Crime Count': [summer_data],
               'Fall Crime Count': [fall_data]
})
#assigning the data from above varibale in a dataframe

#printing the dataframe
season_df

In [ ]:
season_df.T.plot(kind='bar')

In [ ]:
#Q5 Analyze the crimes for each day of the week for all categories of crime.

In [ ]:
# UCR_PART is a reporting classification, not a complete severity ranking.

In [ ]:
#importing required library
import plotly.express as px

#performed group-by on day_of_week, Ucr_part column as we want to acess data on each of the week and each ucr part accordingly.
ucr_day = pd.DataFrame(data = (df.groupby(["DAY_OF_WEEK", "UCR_PART"]).count()["OFFENSE_CODE_GROUP"]).reset_index().values,
                      columns = ["DAY_OF_WEEK", "UCR_PART", "noc"]).sort_values("DAY_OF_WEEK").reset_index(drop = True)

In [ ]:
#setting the x-axis and y-axis along with title and color of the plot
fig3 = px.bar(ucr_day, x = "DAY_OF_WEEK", y = "noc", 
              color = "UCR_PART", title = "UCRs per Day", 
              labels = {"day" : "DAY_OF_WEEK", "noc" : "Number of Crime", "ucr_part" : "UCR_PART"}, text = "noc",
                color_discrete_sequence=["red", "green", "blue", "orange"])

In [ ]:
fig3.update_traces(textposition = "outside")
#assigning the UI for the bar plot
fig3.update_layout(
    font_color="#5642C5",
    title_font_color="#5642C5",
    legend_title_font_color="#5642C5",
    font_size = 14)